# 03 — Model Comparison

Side-by-side comparison of Random Forest and GRU classifiers.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG
from src.metrics import weighted_log_loss, macro_f1, per_class_metrics, plot_confusion_matrix
from src.utils import get_class_name, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Load saved predictions
rf_data = np.load('../../data/rf_test_preds.npz', allow_pickle=True)
rnn_data = np.load('../../data/rnn_test_preds.npz', allow_pickle=True)

class_names = list(rf_data['class_names'])
print(f"Classes: {class_names}")

## 1. Summary Comparison

In [ ]:
models = {
    'Random Forest': rf_data,
    'GRU': rnn_data,
}

comparison = []
for name, data in models.items():
    ll = weighted_log_loss(data['y_true'], data['y_pred_proba'])
    mf1 = macro_f1(data['y_true'], data['y_pred'])
    acc = np.mean(data['y_true'] == data['y_pred'])
    comparison.append({'Model': name, 'Log-Loss': ll, 'Macro F1': mf1, 'Accuracy': acc})

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

## 2. Per-Class F1 Comparison

In [ ]:
rf_metrics = per_class_metrics(rf_data['y_true'], rf_data['y_pred'],
                               {i: class_names[i] for i in range(len(class_names))})
rnn_metrics = per_class_metrics(rnn_data['y_true'], rnn_data['y_pred'],
                                {i: class_names[i] for i in range(len(class_names))})

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(class_names))
width = 0.35

ax.bar(x - width/2, rf_metrics['f1'], width, label='Random Forest', color='#2196F3', alpha=0.8)
ax.bar(x + width/2, rnn_metrics['f1'], width, label='GRU', color='#FF5722', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 — Random Forest vs GRU')
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
save_figure(fig, 'per_class_f1_comparison')
plt.show()

## 3. Where Does Each Model Win?

In [ ]:
delta_f1 = rnn_metrics['f1'].values - rf_metrics['f1'].values

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#FF5722' if d > 0 else '#2196F3' for d in delta_f1]
ax.barh(range(len(class_names)), delta_f1, color=colors)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel('F1 Difference (GRU - RF)')
ax.set_title('F1 Improvement: GRU over RF (positive = GRU wins)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 4. Ensemble Experiment

In [ ]:
# Average predicted probabilities (simple ensemble)
# Note: RF and GRU may have different test sets due to different splitting.
# This only works if they share the same test split.
if np.array_equal(rf_data['y_true'], rnn_data['y_true']):
    ensemble_proba = 0.5 * rf_data['y_pred_proba'] + 0.5 * rnn_data['y_pred_proba']
    ensemble_pred = ensemble_proba.argmax(axis=1)
    ensemble_ll = weighted_log_loss(rf_data['y_true'], ensemble_proba)
    ensemble_f1 = macro_f1(rf_data['y_true'], ensemble_pred)
    print(f"Ensemble Log-Loss: {ensemble_ll:.4f}")
    print(f"Ensemble Macro F1: {ensemble_f1:.4f}")
else:
    print("Test sets differ between models — cannot ensemble directly.")

## 5. Side-by-Side Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

plot_confusion_matrix(rf_data['y_true'], rf_data['y_pred'],
                      class_names, normalize=True, ax=axes[0])
axes[0].set_title('Random Forest')

plot_confusion_matrix(rnn_data['y_true'], rnn_data['y_pred'],
                      class_names, normalize=True, ax=axes[1])
axes[1].set_title('GRU')

plt.tight_layout()
save_figure(fig, 'model_comparison')
plt.show()